> ## SOLUTION / ANSWER KEY &mdash; Lab 9.8
> This is the **completed** notebook (all `___` blanks filled). For the student version, open
> [`../lab-08-audit-trail.ipynb`](../lab-08-audit-trail.ipynb). Trainer use &mdash; or self-check after you've tried it yourself.

# Lab 9.8 &mdash; The Audit Trail

**Level:** Intermediate &nbsp;|&nbsp; **Est. time:** 30 min &nbsp;|&nbsp; **Day 5 &middot; Module 9 &mdash; Agents in Finance, Healthcare &amp; Cybersecurity**

### What you'll do
- Record each step of the run with its detail and source
- Read back the ordered trail for replay
- Check every figure step carries a source

> **How this lab works (near-real):** read the **Concept**, fill the real `___` blanks in **Build it** (the real grounding / citation / compute logic, or the real `create_agent` wiring), then **Run it** and read the output &mdash; and, for the agent labs, the real **message trace**. Finish with an open **Your turn**. There is **no auto-grader**; the goal is a working, grounded insight agent you can read.

> **Framework note:** these labs use the **real** LangChain 1.x (`langchain`, `langchain-core`, `langchain-groq`). The insight agent is a **real** `create_agent` driven by a **real hosted model** (`ChatGroq("openai/gpt-oss-20b")`, key in `.env` as `GROQ_API_KEY`). You are building the **financial-report insight agent** &mdash; the client's Lab 5.1. It is **informational only**: it grounds &amp; **cites** every figure, gives **no advice**, and has **no trade tool** (`place_trade` is defined but never bound &mdash; a human analyst decides). A `@tool` must **catch its own errors and return a string** &mdash; a tool that raises can abort the whole run. If `GROQ_API_KEY` is unset, the run cells print how to set it instead of crashing.

**Reference:** [Module 9 slides &mdash; Auditability: structure & the trail](../../presentation/day5-module9-agents-in-industry.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 9 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, pathlib
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True), override=True)   # GROQ_API_KEY (the Day-5 provider)

WORK = os.path.join(os.environ.get("TEMP") or os.environ.get("TMP") or "/tmp", "biaa-lab-09-08")
os.makedirs(WORK, exist_ok=True)

def groq_ready():
    """True if a GROQ_API_KEY is set. Day-5 labs call a REAL hosted model (Groq)."""
    return bool(os.environ.get("GROQ_API_KEY"))

from langchain_groq import ChatGroq
# Day-5 provider: a REAL hosted model. openai/gpt-oss-20b is a reliable tool-caller via create_agent.
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0) if groq_ready() else None

def print_trace(result):
    """Print a REAL agent message trace: tool calls the model made, tool observations, final answer."""
    for m in result["messages"]:
        for tc in (getattr(m, "tool_calls", None) or []):
            print("TOOL CALL:", tc["name"], tc["args"])
        if type(m).__name__ == "ToolMessage":
            print("OBS:", str(m.content)[:200])
        elif str(getattr(m, "content", "")).strip():
            print(type(m).__name__, ":", str(m.content)[:300])

if groq_ready():
    print("GROQ_API_KEY set | model: openai/gpt-oss-20b | WORK:", WORK)
else:
    print("GROQ_API_KEY NOT set -- add it to .env (free at console.groq.com).")
    print("(The 'Run it for real' cells will print this note instead of crashing.)  WORK:", WORK)

## Concept
In a regulated domain you must be able to **prove** how the agent reached its conclusion (deck slides 15,
19). The **audit trail** logs the whole run &mdash; the trigger, every figure pulled and its source, the
reasoning, the output, the human decision &mdash; so the run is **replayable** and every figure is
**traceable**. It's also how you debug and improve the agent. (In Lab 11 the real agent's **message trace**
is exactly this, produced for free by `create_agent`.)

In [ ]:
# One entry per step: what happened, and (for a figure) where it came from.
print("an entry:", {"step": "figure", "detail": "revenue=120.0M", "source": "p4, income stmt"})

## Build it
Complete `AuditTrail`: record entries, read back the steps, and check every figure is sourced. Then run
the cell.

In [ ]:
class AuditTrail:
    def __init__(self):
        self.entries = []
    def record(self, step, detail, source=None):
        self.entries.append({"step": step, "detail": detail, "source": source})
    def steps(self):
        return [e['step'] for e in self.entries]
    def fully_sourced(self):
        # every FIGURE step must carry a source (analysis/decision steps need not)
        return all(e["source"] for e in self.entries if e["step"] == "figure")

try:
    t = AuditTrail()
    t.record("trigger", "analyze Q3 report")
    t.record("figure", "revenue=120.0M", "p4, income stmt")
    t.record("figure", "net_income=9.0M", "p4, income stmt")
    t.record("output", "summary + needs_review")
    print("steps        :", t.steps())
    print("fully sourced:", t.fully_sourced())
except Exception as e:
    print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## What to notice
- The trail preserves **order**, so the run is replayable step by step &mdash; what a regulator asks for.
- `fully_sourced` checks only **figure** steps carry a source (a "trigger" or "output" step needn't).
- The real agent gives you this automatically: its list of `AIMessage`/`ToolMessage` **is** the trail (Lab 11).

## Your turn (open task &mdash; no grader)
Add a step to the trail with **no** source for a figure and confirm `fully_sourced` turns False; then add
a "human_decision" step and read the full ordered trail. **What good looks like:** any unsourced figure is
caught, and the trail reads as a replayable record of the whole run.

In [ ]:
# --- Reference answer (ONE good way to do the 'Your turn' task -- compare with your own) ---
t = AuditTrail()
t.record("trigger", "analyze Q3 report")
t.record("figure", "revenue=120.0M", "p4, income stmt")
t.record("figure", "ebitda=15.0M", None)              # unsourced figure -- must be caught
print("fully sourced?", t.fully_sourced())            # False
t.record("human_decision", "analyst approved summary for filing")
for e in t.entries:                                    # the full ordered, replayable trail
    print(e["step"], "|", e["detail"], "|", e["source"])

---
### Key takeaway
The audit trail makes the whole run replayable and every figure traceable -- what a regulator needs and what lets you debug the agent. Structured claims make each answer checkable; the trail makes the run accountable.

[&#8592; All Module 9 labs](./index.html) &nbsp;&middot;&nbsp; [Module 9 slides](../../presentation/day5-module9-agents-in-industry.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>